# Maia Lite 0.3.0 — A100 80 GB — Resume verification 5 → 10

**Objetivo:** restaurar o checkpoint sintético de 5 atualizações e continuar até 10, preservando pesos, otimizador e RNG. **Não usa o Maia Corpus v8.**

1. No Colab, selecione a GPU A100 80 GB. Reinicie a sessão (não apenas a execução de uma célula).
2. Execute todas as células em ordem. A execução **falha intencionalmente** se o checkpoint 5 não estiver presente ou se a retomada não iniciar no update 5.
3. Ao final, verifique `checkpoint_0000010.pt` e o relatório `resume_verification_5_to_10.json`.

**Importante:** execute este notebook apenas uma vez. Não apague o checkpoint 5 nem os dados do Drive. O estado do dataloader real ainda não está integrado; este é um teste com entradas sintéticas determinísticas.


In [ ]:
import os, json, math, time, random, hashlib, gc
from pathlib import Path
from dataclasses import asdict
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', force_remount=False)
assert Path('/content/drive/MyDrive').is_dir(), 'Google Drive indisponível'
assert torch.cuda.is_available(), 'Selecione GPU A100 no Colab'
print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.get_device_properties(0).total_memory >= 70 * 2**30, 'GPU não parece A100 80 GB'
print('PyTorch:', torch.__version__, 'BF16:', torch.cuda.is_bf16_supported())
assert torch.cuda.is_bf16_supported()
SEED=20261105
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision('high')

In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss
model=MaiaGPT(cfg)
params=sum(p.numel() for p in model.parameters())
print('Parameters:',format(params,','),'(~%.3f B)'%(params/1e9))
assert 480_000_000<=params<=530_000_000
assert model.head.weight.data_ptr()==model.tok.weight.data_ptr()

In [ ]:
# Identidade do ensaio e política de autorização
PROJECT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
AUDIT_ROOT=PROJECT/'corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full/recovery_audit_v4'
PROVENANCE=AUDIT_ROOT/'production_resume_v2/wikimedia_rights_review/provenance_audit/provenance_summary.json'
assert PROVENANCE.is_file(), 'Relatório de proveniência ausente'
prov=json.loads(PROVENANCE.read_text())
assert prov.get('scanned_chunks')==650 and prov.get('status')=='COMPLETE_PENDING_HUMAN_REVIEW'
assert prov.get('metadata_value_counts',{}).get('eligible_for_training',{}).get('false')==313162
print('Proveniência:',prov['status'], 'registros:',prov['scanned_records'])
print('Treinamento de produção com corpus real: NÃO AUTORIZADO neste notebook')
# Não deduzir autorização a partir da existência de um arquivo.
MODE='synthetic_validation'
assert MODE=='synthetic_validation'

In [ ]:
# Configuração candidata do benchmark
MICROBATCH=8
ACCUM=4
SEQ=2048
TOKENS_PER_UPDATE=MICROBATCH*ACCUM*SEQ
TARGET_TOKENS=20*params
TARGET_UPDATES=math.ceil(TARGET_TOKENS/TOKENS_PER_UPDATE)
TEST_UPDATES=5  # teste sintético curto; aumente para 20 para estabilidade
WARMUP=2000
LR=3e-4   # candidato, não taxa final validada para treino real
MIN_LR=3e-5
WEIGHT_DECAY=0.1
GRAD_CLIP=1.0
CKPT_EVERY=5
RUN_DIR=PROJECT/'pretraining_05b_synthetic_validation_v1'
RUN_DIR.mkdir(parents=True,exist_ok=True)
RUN_ID='maia_05b_synthetic_v1'
IDENTITY={'run_id':RUN_ID,'mode':MODE,'model_config':asdict(cfg),'parameters':params,'microbatch':MICROBATCH,'accum':ACCUM,'seq':SEQ,'seed':SEED,'target_tokens':TARGET_TOKENS,'target_updates':TARGET_UPDATES}
print('Tokens/update:',TOKENS_PER_UPDATE,'Chinchilla updates:',TARGET_UPDATES)

In [ ]:
# Checkpoint de estado completo, atômico, com retomada. Dados sintéticos são determinísticos por (update,microstep).
import io
import copy

def lr_at(step):
    if step < WARMUP: return LR*(step+1)/WARMUP
    progress=min(1.,max(0.,(step-WARMUP)/max(1,TARGET_UPDATES-WARMUP)))
    return MIN_LR+0.5*(LR-MIN_LR)*(1+math.cos(math.pi*progress))

def synthetic_batch(update,microstep):
    g=torch.Generator(device='cpu').manual_seed(SEED+update*1009+microstep)
    # Gera uma janela adicional para prever o token seguinte
    ids=torch.randint(0,cfg.vocab_size,(MICROBATCH,SEQ+1),generator=g)
    return ids[:,:-1].to('cuda',non_blocking=True),ids[:,1:].to('cuda',non_blocking=True)

def save_checkpoint(step, optimizer):
    target=RUN_DIR/f'checkpoint_{step:07d}.pt'
    tmp=target.with_suffix('.pt.tmp')
    state={'identity':IDENTITY,'step':step,'tokens_seen':step*TOKENS_PER_UPDATE,
           'model':model.state_dict(),'optimizer':optimizer.state_dict(),
           'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),
           'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),
           'saved_utc':__import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat()}
    torch.save(state,tmp)
    os.replace(tmp,target)
    print('Checkpoint completo:',target,'GiB:',round(target.stat().st_size/2**30,2))
    return target

model=model.to('cuda')
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,betas=(0.9,0.95),eps=1e-8,weight_decay=WEIGHT_DECAY)
start_step=0
checkpoints=sorted(RUN_DIR.glob('checkpoint_*.pt'))
assert (RUN_DIR/'checkpoint_0000005.pt').is_file(), 'Checkpoint 5 ausente; não iniciar do zero'
assert not (RUN_DIR/'checkpoint_0000010.pt').exists(), 'Checkpoint 10 já existe; teste já executado. Não sobrescrever.'
assert checkpoints[-1].name == 'checkpoint_0000005.pt', 'Último checkpoint não é o 5; confira a pasta antes de continuar'
if checkpoints:
    latest=checkpoints[-1]
    print('Retomando:',latest)
    state=torch.load(latest,map_location='cpu',weights_only=False)
    assert state['identity']==IDENTITY, 'Checkpoint incompatível; não continuar'
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    # Transferir estados do AdamW à GPU se necessário
    for group in optimizer.state.values():
        for k,v in group.items():
            if torch.is_tensor(v):group[k]=v.to('cuda')
    random.setstate(state['python_rng']);np.random.set_state(state['numpy_rng'])
    torch.set_rng_state(state['torch_rng']);torch.cuda.set_rng_state_all(state['cuda_rng'])
    start_step=int(state['step'])
    del state;gc.collect()
print('Retomada a partir do update:',start_step)
assert start_step == 5, f'Era esperado retomar do update 5, encontrado {start_step}'
assert optimizer.state, 'Estado do AdamW vazio após restauração'
assert all(torch.isfinite(p).all().item() for p in model.parameters()), 'Pesos não finitos'
print('RESTORE VERIFIED: model, AdamW, RNG, identity, step=5')

In [ ]:
# EXECUÇÃO DE TESTE: 5 atualizações, apenas dados sintéticos.
assert MODE=='synthetic_validation'
model.train()
torch.cuda.reset_peak_memory_stats()
end_step=start_step+TEST_UPDATES
for step in range(start_step,end_step):
    t0=time.perf_counter()
    optimizer.zero_grad(set_to_none=True)
    for group in optimizer.param_groups:group['lr']=lr_at(step)
    loss_sum=0.0
    for micro in range(ACCUM):
        x,y=synthetic_batch(step,micro)
        with torch.autocast('cuda',dtype=torch.bfloat16):
            _,loss=model(x,y)
        (loss/ACCUM).backward()
        loss_sum+=float(loss.detach())/ACCUM
        del x,y,loss
    norm=torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
    assert math.isfinite(loss_sum) and math.isfinite(float(norm)), 'Loss/gradiente não finito'
    optimizer.step()
    torch.cuda.synchronize()
    elapsed=time.perf_counter()-t0
    print(f'update={step+1} loss={loss_sum:.4f} grad_norm={float(norm):.3f} lr={lr_at(step):.3g} tokens/s={TOKENS_PER_UPDATE/elapsed:.0f} peak_GiB={torch.cuda.max_memory_allocated()/2**30:.1f}',flush=True)
    if (step+1)%CKPT_EVERY==0 or step+1==end_step:
        save_checkpoint(step+1,optimizer)
assert end_step == 10
assert (RUN_DIR/'checkpoint_0000010.pt').is_file(), 'Checkpoint 10 não foi criado'
report={'status':'RESUME_5_TO_10_COMPLETED','restored_from_update':start_step,'final_update':end_step,'checkpoint_5':str(RUN_DIR/'checkpoint_0000005.pt'),'checkpoint_10':str(RUN_DIR/'checkpoint_0000010.pt'),'tokens_seen':end_step*TOKENS_PER_UPDATE,'mode':MODE,'identity':IDENTITY}
report_path=RUN_DIR/'resume_verification_5_to_10.json'
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('RESUME VERIFIED: 5 -> 10')
print('Checkpoint:',report['checkpoint_10'])
print('Report:',report_path)
print('VALIDAÇÃO SINTÉTICA CONCLUÍDA. Nenhum dado do corpus real foi usado.')

## Próxima integração (não executada)

Para produção: (1) documentar aprovação por fonte; (2) congelar tokenizer e hash; (3) validar formatos dos shards de tokens, splits e hashes; (4) montar um dataloader determinístico que registre ordem/offset de blocos e o estado no checkpoint; (5) executar ensaio com dados elegíveis; (6) só então programar 154.733 updates. Este notebook **não contém um botão que ignora `eligible_for_training=false`**. A quantidade de 6,94 B tokens materializados não representa tokens aprovados; a meta de 10,14 B exigirá aproximadamente 1,46 exposições sobre o conjunto atual, caso ele venha a ser aprovado.

## Interpretação

`RESTORE VERIFIED` confirma que o checkpoint 5 foi carregado com identidade, parâmetros, AdamW e RNG. `RESUME VERIFIED: 5 -> 10` confirma cinco atualizações adicionais e o arquivo de checkpoint 10. Não é um teste de equivalência bit a bit com uma execução ininterrupta; para isso seria necessário um teste A/B controlado. Também não valida a retomada do dataloader do corpus real.
